# Deep Dive: Multi-Agent Orchestration & TDD with AI

## 📋 Summary

Complex systems often hit two distinct scaling walls:
1. **Single-agent cognitive overload**: One prompt trying to research, analyze, and format simultaneously loses focus and runs sequentially. **Multi-Agent Orchestration** (Lesson 10) solves this with an orchestrator delegating to scoped subagents (sequentially or in parallel with `ThreadPoolExecutor` / `Promise.all`) across mixed model tiers.
2. **Unverified AI-generated code**: Generating code first and testing later hides subtle edge cases. **AI-Accelerated TDD** (Lesson 11) flips the order — write clear specs, let AI generate a comprehensive **Red** test suite, audit it for domain edge cases AI misses, then implement to **Green** and **Refactor**.

In this notebook (covering **Lessons 10–11**), you will:
- Build a **Travel Disruption Coordinator** (Orchestrator + 3 Haiku/Sonnet specialists in sequential and parallel fan-out topologies)
- Practice **Red-Green-Refactor TDD** on a fresh **Event Ticket Reservation Engine**

### Key concepts in this notebook

| # | Concept | Complexity |
|---|---|---|
| 1 | Orchestrator + scoped specialist subagents (`AgentDefinition` pattern) | 🟢 Beginner |
| 2 | Sequential pipeline vs parallel fan-out + merge latency comparison | 🟡 Intermediate |
| 3 | Preventing tool leakage & capping subagent turns/cost | 🟡 Intermediate |
| 4 | Spec-first AI TDD: writing structured requirements & Red tests | 🟡 Intermediate |
| 5 | Auditing AI-generated tests for domain edge cases (Red → Green → Refactor) | 🔴 Advanced |
| 6 | Hands-on orchestration & TDD exercises | 🏋️ Practice |

> **How to use this notebook:** Run cells in order. Cells with `# TODO` are exercises.

## 0. Setup

In [ ]:
%pip install -q anthropic pydantic

import concurrent.futures
import os
import time
from dataclasses import dataclass
from getpass import getpass
from pydantic import BaseModel, Field

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    if not os.environ.get("ANTHROPIC_API_KEY"):
        os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your ANTHROPIC_API_KEY: ")

import anthropic

client = anthropic.Anthropic()
MODEL_HAIKU = "claude-haiku-4-5-20251001"
MODEL_SONNET = "claude-sonnet-4-5-20250929"
print("Setup complete.")

## 1. Defining Scoped Specialist Subagents 🟢

In Lesson 10 (`research-orchestrator.ts` and `sales-qualifier.ts`), every subagent is defined with four properties (`AgentDefinition`):
- `name` & `description` (when the orchestrator should invoke it)
- `prompt` (focused specialist instructions)
- `tools` (least privilege — e.g., analyzer and scorer get `tools: []`)
- `model` (`haiku` for bounded extraction/analysis, `sonnet` for final synthesis)

Let's model our **Travel Disruption Coordinator** with 3 specialist subagents:
1. `rebooking-scout` (Haiku): finds alternate flight options from disruption details
2. `policy-advisor` (Haiku): checks airline delay/compensation rules
3. `passenger-concierge` (Sonnet): synthesizes a structured, empathetic action plan

In [ ]:
@dataclass
class SubagentDef:
    name: str
    description: str
    system_prompt: str
    model: str


SUBAGENTS = {
    "rebooking-scout": SubagentDef(
        name="rebooking-scout",
        description="Evaluates alternate flight routes and arrival trade-offs.",
        system_prompt="You are a flight rebooking specialist. Given a cancelled flight and candidate alternatives, rank the top 2 options with connection-risk notes in <= 80 words.",
        model=MODEL_HAIKU,
    ),
    "policy-advisor": SubagentDef(
        name="policy-advisor",
        description="Determines hotel voucher, meal credit, and statutory compensation eligibility.",
        system_prompt="You are an aviation passenger-rights specialist. State likely entitlements (meal voucher, hotel, EU261/DOT refund rights) based on the delay reason and duration in <= 80 words.",
        model=MODEL_HAIKU,
    ),
    "passenger-concierge": SubagentDef(
        name="passenger-concierge",
        description="Synthesizes specialist findings into a concise passenger briefing.",
        system_prompt="You are a travel concierge orchestrator. Combine the rebooking and policy reports into a clear 3-step action plan for the traveler.",
        model=MODEL_SONNET,
    ),
}


def run_subagent(agent_key: str, user_input: str) -> tuple[str, float]:
    spec = SUBAGENTS[agent_key]
    t0 = time.perf_counter()
    resp = client.messages.create(
        model=spec.model,
        max_tokens=300,
        system=spec.system_prompt,
        messages=[{"role": "user", "content": user_input}],
    )
    ms = (time.perf_counter() - t0) * 1000
    return resp.content[0].text.strip(), ms

## 2. Sequential vs Parallel Fan-Out & Merge 🟡

When `rebooking-scout` and `policy-advisor` do **not** depend on each other's output, running them concurrently cuts wall-clock latency nearly in half while feeding both outputs into `passenger-concierge`.

In [ ]:
DISRUPTION_CASE = """
Flight UX-402 (Madrid -> Boston) cancelled due to mechanical issue (airline fault).
Expected delay before next departure: 9 hours overnight.
Candidate alternatives:
- Option A: Direct flight tomorrow 09:00 (arrives 11:30 local)
- Option B: Tonight via Lisbon (45-min connection, arrives 06:15 local)
- Option C: Tomorrow via London (3h layover, arrives 16:00 local)
"""

# 1) Sequential execution
t_seq0 = time.perf_counter()
rebook_seq, ms1 = run_subagent("rebooking-scout", DISRUPTION_CASE)
policy_seq, ms2 = run_subagent("policy-advisor", DISRUPTION_CASE)
seq_workers_ms = (time.perf_counter() - t_seq0) * 1000

# 2) Parallel fan-out execution
t_par0 = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
    f_rebook = pool.submit(run_subagent, "rebooking-scout", DISRUPTION_CASE)
    f_policy = pool.submit(run_subagent, "policy-advisor", DISRUPTION_CASE)
    rebook_par, _ = f_rebook.result()
    policy_par, _ = f_policy.result()
par_workers_ms = (time.perf_counter() - t_par0) * 1000

# 3) Orchestrator merge (Sonnet)
merge_input = f"Rebooking Report:\n{rebook_par}\n\nPolicy Report:\n{policy_par}"
final_plan, merge_ms = run_subagent("passenger-concierge", merge_input)

print(f"Workers sequential: {seq_workers_ms:.0f} ms | Workers parallel: {par_workers_ms:.0f} ms | Merge: {merge_ms:.0f} ms")
print("\n=== FINAL PASSENGER ACTION PLAN ===")
print(final_plan)

### 🧐 Observation
- Compare `seq_workers_ms` vs `par_workers_ms`. As you add 3, 4, or 5 independent specialists, parallel fan-out keeps worker latency bounded by the *slowest single worker* rather than the *sum* of all workers.
- Notice the orchestrator (`passenger-concierge`) never duplicates the subagents' work — it only merges their focused outputs.

## 3. AI-Accelerated TDD: Spec → Red Tests → Green → Refactor 🟡

Lesson 11 teaches a disciplined 3-phase TDD loop with AI:
1. **Write a crisp spec first** (inputs, outputs, validation rules, error conditions).
2. **Generate tests BEFORE implementation (RED)**: Ask Claude to write a comprehensive test suite.
3. **Human edge-case audit**: AI reliably tests happy paths and basic validation, but **humans must check** multi-step state transitions, cumulative capacity limits, rounding/currency precision, and security boundaries.
4. **Implement to GREEN, then REFACTOR** without breaking any test.

Let's apply this to a fresh **Event Ticket Reservation Engine** using the **Result-Object Pattern** (`{"success": bool, "error": ...}`) from Lesson 11!

In [ ]:
class TicketReservationEngine:
    """Deliberately minimal starter implementation (has intentional edge-case gaps!)."""

    def __init__(self, capacity: int, price_usd: float):
        self.capacity = capacity
        self.price_usd = price_usd
        self.reservations: dict[str, int] = {}

    def available_seats(self) -> int:
        return self.capacity - sum(self.reservations.values())

    def reserve(self, user_id: str, qty: int) -> dict:
        if qty <= 0:
            return {"success": False, "error": "Quantity must be positive"}
        if qty > self.available_seats():
            return {"success": False, "error": "Insufficient seats"}
        self.reservations[user_id] = self.reservations.get(user_id, 0) + qty
        total = round(qty * self.price_usd, 2)
        return {"success": True, "user_id": user_id, "qty": qty, "total_usd": total}

    def cancel(self, user_id: str) -> dict:
        if user_id not in self.reservations:
            return {"success": False, "error": "No reservation found"}
        released = self.reservations.pop(user_id)
        return {"success": True, "released_qty": released}


# Run baseline tests (GREEN for happy path, exposes edge-case requirements)
def run_baseline_tests():
    eng = TicketReservationEngine(capacity=10, price_usd=49.99)
    r1 = eng.reserve("alice", 3)
    assert r1["success"] and r1["total_usd"] == 149.97
    assert eng.available_seats() == 7
    assert not eng.reserve("bob", 8)["success"]
    assert not eng.reserve("bob", 0)["success"]
    assert eng.cancel("alice")["released_qty"] == 3
    assert eng.available_seats() == 10
    print("All 6 baseline tests PASSED!")


run_baseline_tests()

### 🧐 Observation: What Edge Cases Would AI-Generated Tests Easily Miss?

| Edge Case | Why AI Often Misses It | How to Test It |
|---|---|---|
| Cumulative per-user cap (e.g., max 4 tickets per user across *multiple* calls) | AI tests single-call `reserve("alice", 5)` but forgets `reserve("alice", 3)` + `reserve("alice", 2)` | Two sequential calls for the same `user_id` |
| Non-integer / boolean quantities (`qty=2.5` or `qty=True`) | In Python `isinstance(True, int)` is `True`! | Pass `2.5` and `True` explicitly |
| Floating-point total drift (`3 * 19.99`) | Exact `==` fails without `round(..., 2)` | Assert `round(val, 2)` or `math.isclose` |
| Idempotent / double cancel | Calling `cancel("alice")` twice | Assert second call returns `success=False` cleanly without crashing |

## 4. Failure Modes & Fixes 🔴

| Failure Mode | Why It Happens | Fix |
|---|---|---|
| Orchestrator bypasses subagents | Orchestrator was given the same tools (`WebSearch`) as workers | Restrict orchestrator to delegation (`allowedTools: ["Task"]`) |
| Runaway subagent loops | No turn cap on delegated tasks | Set `maxTurns` and explicit tool-call caps in subagent prompts |
| Tautological AI-generated tests | AI writes `assert result is not None` instead of checking exact values | Audit every generated test against the spec's exact invariants |
| State leakage between tests | Shared module-level dict mutated across tests | Provide a `reset()` / fresh instance per test |

## 5. Exercises 🏋️

### Exercise 1: Add a Third Parallel Specialist (`hotel-finder`)
Add a `hotel-finder` subagent (Haiku) to `SUBAGENTS` that recommends an airport hotel given the disruption case, run all 3 workers in parallel with `ThreadPoolExecutor`, and pass all 3 reports to `passenger-concierge`.

In [ ]:
# TODO: Add "hotel-finder" to SUBAGENTS, run rebooking-scout + policy-advisor + hotel-finder
# in parallel, and synthesize the combined result with passenger-concierge.
print("Ready to add hotel-finder subagent.")

### Exercise 2: Red-Green-Refactor — Enforce a Per-User Ticket Cap (Max 4)
1. **RED**: Write 3 tests *first* asserting that:
   - A user cannot reserve `5` tickets in one call
   - A user cannot reserve `3` tickets and then `2` more (cumulative `5 > 4`)
   - Non-integer or boolean `qty` (like `2.5` or `True`) is rejected
2. **GREEN**: Subclass `TicketReservationEngine` as `CappedReservationEngine` and make all 3 tests pass.

In [ ]:
class CappedReservationEngine(TicketReservationEngine):
    MAX_PER_USER = 4

    def reserve(self, user_id: str, qty: int) -> dict:
        # TODO: Reject booleans/non-integers and enforce cumulative MAX_PER_USER cap
        return super().reserve(user_id, qty)


def test_capped_engine():
    eng = CappedReservationEngine(capacity=20, price_usd=25.00)
    # TODO: Write your 3 RED tests here, then implement CappedReservationEngine.reserve to make them GREEN
    print("Write assertions for CappedReservationEngine!")


test_capped_engine()

## 🏁 Summary & Key Takeaways

- **Multi-agent orchestration** works best when subagents have narrow prompts, least-privilege tools, and right-sized models (`haiku` workers + `sonnet` orchestrator).
- **Parallelize independent work:** Fan out independent subagents concurrently and merge once all complete.
- **TDD with AI = Spec + Human Edge-Case Audit:** Let AI accelerate test generation from specs, then verify multi-step state transitions, boundary types, and precision before implementing to Green.

### What's next?
Continue to **`deep-dive-7-evaluating-agents-and-computer-use.ipynb`** to build multi-dimensional agent evaluation harnesses with partial-credit scoring and explore Claude's Computer Use screenshot-action loop.